In [1]:
import os

In [2]:
%pwd

'd:\\AI-ML\\REAL WORLD PROJECTS\\MACHINE LEARNING\\Drinks-Quality-Prediction-System\\Notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\AI-ML\\REAL WORLD PROJECTS\\MACHINE LEARNING\\Drinks-Quality-Prediction-System'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml,create_directories
from mlProject.entity.config_entity import (DataIngestionConfig)

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)->DataIngestionConfig:

        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [7]:
import os
import urllib.request as requests
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size
from pathlib import Path
from mlProject.entity.config_entity import DataTransformatinConfig

class DataIngestion:
    def __init__(self,config:DataTransformatinConfig):
        self.config=config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename,headers=requests.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} Download! With following info: \n {headers}")
        else:
            logger.info(f"File already exists of size : {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):
        unzip_path=self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [8]:
from mlProject.utils.common import read_yaml

config = read_yaml("config/config.yaml")

print(config.data_ingestion)

[2026-08-21 13:37:16,696 : INFO : common: Yaml file : config\config.yaml Loaded Successfully]
{'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://raw.githubusercontent.com/mdzaheerjk/Drinks-Quality-Prediction-System/main/data/Drinks-data.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}


In [9]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-08-21 13:37:16,713 : INFO : common: Yaml file : config\config.yaml Loaded Successfully]
[2026-08-21 13:37:16,715 : INFO : common: Yaml file : params.yaml Loaded Successfully]
[2026-08-21 13:37:16,719 : INFO : common: Yaml file : schema.yaml Loaded Successfully]
[2026-08-21 13:37:16,720 : INFO : common: created directory at : artifacts]
[2026-08-21 13:37:16,722 : INFO : common: created directory at : artifacts/data_ingestion]
[2026-08-21 13:37:16,723 : INFO : 1864487442: File already exists of size : ~23 KB]
